In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 01: fan_health_kpis
-- ══════════════════════════════════════════════════════
-- Single-row scorecard for counter widgets at the top of the dashboard.
-- Each column becomes one big number tile.

SELECT
  -- Total active fan base
  COUNT(*)                                                          AS total_fans,

  -- High risk fans needing immediate attention
  SUM(CASE WHEN c.risk_tier = 'High'   THEN 1 ELSE 0 END)         AS high_risk_fans,

  -- Percentage of base that is high risk
  ROUND(
    SUM(CASE WHEN c.risk_tier = 'High' THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                                                 AS high_risk_pct,

  -- Average churn probability across all fans
  ROUND(AVG(c.churn_probability) * 100, 1)                        AS avg_churn_prob_pct,

  -- Season Core fans (healthiest segment)
  SUM(CASE WHEN s.segment_label = 'Season Core' THEN 1 ELSE 0 END) AS season_core_count,

  -- Jersey Night cohort 30-day return rate
  ROUND(
    SUM(CASE WHEN f.is_jersey_night_cohort = 1
              AND r.returned_30d = TRUE THEN 1 ELSE 0 END)
    * 100.0 /
    NULLIF(SUM(CASE WHEN f.is_jersey_night_cohort = 1 THEN 1 ELSE 0 END), 0),
    1
  )                                                                 AS jersey_night_return_rate_pct,

  -- Total lifetime net revenue across all fans
  ROUND(SUM(c360.revenue_net), 0)                                  AS total_net_revenue

FROM icebase.gold.ml_churn_scores      c
JOIN icebase.gold.ml_fan_segments      s   ON c.customer_id = s.customer_id
JOIN icebase.gold.fan_features         f   ON c.customer_id = f.customer_id
JOIN icebase.gold.customer_360         c360 ON c.customer_id = c360.customer_id
JOIN icebase.gold.retention_cohort     r   ON c.customer_id = r.customer_id;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 02: churn_risk_distribution
-- ══════════════════════════════════════════════════════
-- Risk tier breakdown for bar chart or donut chart.

SELECT
  risk_tier,
  COUNT(*)                                    AS fan_count,
  ROUND(COUNT(*) * 100.0
    / SUM(COUNT(*)) OVER (), 1)              AS pct_of_total,
  ROUND(AVG(churn_probability) * 100, 1)    AS avg_churn_prob_pct,
  CASE risk_tier
    WHEN 'High'   THEN 1
    WHEN 'Medium' THEN 2
    WHEN 'Low'    THEN 3
  END                                         AS sort_order
FROM icebase.gold.ml_churn_scores
GROUP BY risk_tier
ORDER BY sort_order;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 03: segment_summary
-- ══════════════════════════════════════════════════════
-- One row per segment with health metrics — powers a summary table
-- and bar charts showing segment composition.

SELECT
  s.segment_label,
  COUNT(*)                                        AS fan_count,
  ROUND(AVG(c.churn_probability) * 100, 1)       AS avg_churn_prob_pct,
  ROUND(AVG(c360.total_spend), 2)                AS avg_total_spend,
  ROUND(AVG(c360.games_attended), 1)             AS avg_games_attended,
  ROUND(AVG(c360.days_since_last), 0)            AS avg_days_since_last,
  ROUND(AVG(c360.promo_sensitivity), 3)          AS avg_promo_sensitivity,
  ROUND(AVG(c360.tenure_days), 0)                AS avg_tenure_days,
  SUM(CASE WHEN c.risk_tier = 'High'
           THEN 1 ELSE 0 END)                    AS high_risk_count,
  ROUND(
    SUM(CASE WHEN c.risk_tier = 'High' THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                               AS high_risk_pct
FROM icebase.gold.ml_fan_segments      s
JOIN icebase.gold.ml_churn_scores      c   ON s.customer_id = c.customer_id
JOIN icebase.gold.customer_360         c360 ON s.customer_id = c360.customer_id
GROUP BY s.segment_label
ORDER BY avg_churn_prob_pct DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 04: high_risk_watchlist
-- ══════════════════════════════════════════════════════
-- Top 25 high-risk fans worth retaining — sorted by lifetime value.
-- This is the retention team's action list.
-- Shows email so a campaign manager could actually use it.

SELECT
  d.full_name,
  d.email,
  d.acquisition_channel,
  s.segment_label,
  ROUND(c.churn_probability * 100, 1)           AS churn_probability_pct,
  c.risk_tier,
  ROUND(c360.revenue_net, 2)                    AS lifetime_net_spend,
  c360.games_attended,
  c360.days_since_last                          AS days_since_last_game,
  ROUND(c360.promo_sensitivity, 2)              AS promo_sensitivity,
  CASE
    WHEN c360.promo_sensitivity > 0.5 THEN 'Offer discount'
    WHEN c360.games_attended > 5      THEN 'Loyalty reward'
    ELSE 'Re-engagement email'
  END                                           AS recommended_action
FROM icebase.gold.ml_churn_scores      c
JOIN icebase.gold.ml_fan_segments      s   ON c.customer_id = s.customer_id
JOIN icebase.gold.customer_360         c360 ON c.customer_id = c360.customer_id
JOIN icebase.silver.dim_customer       d   ON c.customer_id = d.customer_id
WHERE
  c.risk_tier = 'High'
  AND c360.games_attended >= 2
  AND c360.revenue_net >= 50
ORDER BY c360.revenue_net DESC
LIMIT 25;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 05: jersey_night_cohort
-- ══════════════════════════════════════════════════════
-- Side-by-side comparison of Jersey Night cohort vs general fan base.
-- The "good story" — did the event convert fans?

SELECT
  CASE WHEN f.is_jersey_night_cohort = 1
       THEN 'Jersey Night Cohort'
       ELSE 'General Fan Base'
  END                                           AS cohort,
  COUNT(*)                                      AS fans,
  ROUND(AVG(c.churn_probability) * 100, 1)     AS avg_churn_prob_pct,
  SUM(CASE WHEN r.returned_30d = TRUE
           THEN 1 ELSE 0 END)                  AS returned_within_30d,
  ROUND(
    SUM(CASE WHEN r.returned_30d = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS return_30d_rate_pct,
  ROUND(AVG(c360.revenue_net), 2)              AS avg_lifetime_spend,
  SUM(CASE WHEN c.risk_tier = 'High'
           THEN 1 ELSE 0 END)                  AS high_risk_count
FROM icebase.gold.fan_features         f
JOIN icebase.gold.ml_churn_scores      c   ON f.customer_id = c.customer_id
JOIN icebase.gold.retention_cohort     r   ON f.customer_id = r.customer_id
JOIN icebase.gold.customer_360         c360 ON f.customer_id = c360.customer_id
GROUP BY f.is_jersey_night_cohort
ORDER BY f.is_jersey_night_cohort DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 06: promo_sensitivity_dist
-- ══════════════════════════════════════════════════════
-- Promo sensitivity distribution by segment — shows the "bad story"
-- visually: Promo Hunter and Lapsed have high sensitivity.

SELECT
  s.segment_label,
  CASE
    WHEN f.promo_sensitivity < 0.1  THEN '0.0–0.1 (Never uses promos)'
    WHEN f.promo_sensitivity < 0.3  THEN '0.1–0.3 (Rare)'
    WHEN f.promo_sensitivity < 0.5  THEN '0.3–0.5 (Occasional)'
    WHEN f.promo_sensitivity < 0.7  THEN '0.5–0.7 (Frequent)'
    ELSE                                  '0.7–1.0 (Promo dependent)'
  END                                           AS sensitivity_bucket,
  COUNT(*)                                      AS fan_count,
  ROUND(AVG(f.promo_sensitivity), 3)            AS avg_sensitivity
FROM icebase.gold.fan_features         f
JOIN icebase.gold.ml_fan_segments      s ON f.customer_id = s.customer_id
GROUP BY s.segment_label, sensitivity_bucket
ORDER BY s.segment_label, avg_sensitivity;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 07: acquisition_channel_health
-- ══════════════════════════════════════════════════════
-- Which channels acquired fans who actually stuck around?
-- This surfaces the "walk-up buyers disappear" bad story.

SELECT
  d.acquisition_channel,
  COUNT(*)                                      AS fans_acquired,
  ROUND(AVG(c.churn_probability) * 100, 1)     AS avg_churn_prob_pct,
  ROUND(AVG(c360.games_attended), 1)           AS avg_games_attended,
  ROUND(AVG(c360.revenue_net), 2)              AS avg_lifetime_spend,
  ROUND(
    SUM(CASE WHEN r.returned_30d = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS return_30d_rate_pct,
  SUM(CASE WHEN c.risk_tier = 'High'
           THEN 1 ELSE 0 END)                  AS high_risk_count
FROM icebase.silver.dim_customer       d
JOIN icebase.gold.ml_churn_scores      c   ON d.customer_id = c.customer_id
JOIN icebase.gold.customer_360         c360 ON d.customer_id = c360.customer_id
JOIN icebase.gold.retention_cohort     r   ON d.customer_id = r.customer_id
GROUP BY d.acquisition_channel
ORDER BY avg_lifetime_spend DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 08: churn_by_season_phase
-- ══════════════════════════════════════════════════════
-- Churn flag rate by which season phase the fan was acquired in.
-- Shows the hot-start acquisition quality problem clearly.

SELECT
  r.acquisition_phase,
  COUNT(*)                                      AS fans,
  SUM(CASE WHEN r.churn_flag = TRUE 
           THEN 1 ELSE 0 END)                  AS churned_count,
  ROUND(
    SUM(CASE WHEN r.churn_flag = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS churn_rate_pct,
  ROUND(AVG(c360.games_attended), 1)           AS avg_games_attended,
  ROUND(
    SUM(CASE WHEN r.returned_30d = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS return_30d_rate_pct,
  CASE r.acquisition_phase
    WHEN 'hot_start'    THEN 1
    WHEN 'slump'        THEN 2
    WHEN 'late_push'    THEN 3
    WHEN 'jersey_night' THEN 4
    WHEN 'post_jersey'  THEN 5
    ELSE 6
  END                                           AS phase_order
FROM icebase.gold.retention_cohort     r
JOIN icebase.gold.customer_360         c360 ON r.customer_id = c360.customer_id
WHERE r.acquisition_phase IS NOT NULL
GROUP BY r.acquisition_phase
ORDER BY phase_order;